# 01 — Exploratory Data Analysis & Preprocessing

This notebook kicks off the SaaS Churn Intelligence project by loading the raw Telco churn dataset, understanding its structure, visualizing key patterns, and producing a clean, feature-rich dataset for downstream modelling.

**Goals for this notebook:**
- Understand the shape and quality of the raw data
- Identify churn patterns across contract type, tenure, and charges
- Run the preprocessing pipeline to engineer features
- Save a processed dataset for use in all subsequent notebooks

## 1. Setup & Data Loading

We first add the project root to `sys.path` so that our `src` modules are importable, then generate the raw data file if it doesn't already exist on disk.

In [ ]:
import sys
import os
import subprocess

sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.preprocessing import load_raw, clean, add_features, preprocess, get_model_features

# Suppress non-critical warnings to keep output clean
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

In [ ]:
# Generate the raw data file if it doesn't exist yet
raw_path = '../data/raw/telco_churn.csv'

if not os.path.exists(raw_path):
    print('Raw data not found — generating now...')
    subprocess.run(['python', '../data/generate_data.py'])
    print('Data generation complete.')
else:
    print(f'Raw data found at: {raw_path}')

# Load raw data using our src module
df_raw = load_raw(raw_path)
print(f'\nLoaded {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.')

## 2. Initial Data Inspection

Before doing anything analytical, it's important to understand what we're working with: column types, null counts, and the basic distributional properties of each numeric variable. This quick scan often surfaces data quality issues that would silently corrupt downstream models.

In [ ]:
print('=== Shape ===')
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')

print('\n=== Column dtypes ===')
print(df_raw.dtypes)

In [ ]:
# Summary statistics for all numeric columns
df_raw.describe(include='all').T

In [ ]:
# Missing value audit — critical before any modelling work
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

if missing_df.empty:
    print('No missing values detected. Dataset is complete.')
else:
    print('Columns with missing values:')
    display(missing_df)

## 3. Overall Churn Rate

The headline churn rate anchors everything that follows. In SaaS, even a 2–3 percentage point difference in annual churn can mean millions of dollars in lifetime value. Let's see where this dataset sits.

In [ ]:
churn_dist = df_raw['Churn'].value_counts(normalize=True).mul(100).round(2)
print('Churn distribution (%):')
print(churn_dist.to_string())

churn_count = df_raw['Churn'].value_counts()
print(f'\nTotal customers: {len(df_raw):,}')
print(f'Churned: {churn_count.get("Yes", churn_count.get(1, 0)):,}')
print(f'Retained: {churn_count.get("No", churn_count.get(0, 0)):,}')

## 4. Visualizing Churn Patterns

Raw churn rate is a starting point, but the real signal lies in *which* customers churn. We'll examine three key dimensions:
1. **Contract type** — month-to-month customers are far more likely to leave
2. **Tenure band** — newer customers tend to churn at higher rates (the early-period 'escape' phenomenon)
3. **Monthly charges** — higher-paying customers may have different churn dynamics

In [ ]:
# Churn rate by Contract type
churn_col = 'Churn'
contract_col = 'Contract'

# Normalise the churn column to binary 0/1 if needed
df_plot = df_raw.copy()
if df_plot[churn_col].dtype == object:
    df_plot['churn_binary'] = (df_plot[churn_col] == 'Yes').astype(int)
else:
    df_plot['churn_binary'] = df_plot[churn_col].astype(int)

contract_churn = (
    df_plot.groupby(contract_col)['churn_binary']
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={'churn_binary': 'churn_rate_pct'})
    .sort_values('churn_rate_pct', ascending=False)
)

fig = px.bar(
    contract_churn,
    x=contract_col,
    y='churn_rate_pct',
    color='churn_rate_pct',
    color_continuous_scale='Reds',
    text='churn_rate_pct',
    title='Churn Rate by Contract Type',
    labels={contract_col: 'Contract Type', 'churn_rate_pct': 'Churn Rate (%)'}
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, plot_bgcolor='white', height=400)
fig.show()

In [ ]:
# Churn rate by tenure band
tenure_col = 'tenure'

df_plot['tenure_band'] = pd.cut(
    df_plot[tenure_col],
    bins=[0, 6, 12, 24, 36, 48, 72],
    labels=['0–6 mo', '7–12 mo', '13–24 mo', '25–36 mo', '37–48 mo', '49–72 mo']
)

tenure_churn = (
    df_plot.groupby('tenure_band')['churn_binary']
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={'churn_binary': 'churn_rate_pct'})
)

fig2 = px.bar(
    tenure_churn,
    x='tenure_band',
    y='churn_rate_pct',
    color='churn_rate_pct',
    color_continuous_scale='Blues',
    text='churn_rate_pct',
    title='Churn Rate by Tenure Band',
    labels={'tenure_band': 'Tenure Band', 'churn_rate_pct': 'Churn Rate (%)'}
)
fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig2.update_layout(coloraxis_showscale=False, plot_bgcolor='white', height=400)
fig2.show()

In [ ]:
# Monthly charges distribution overlaid by churn status
charges_col = 'MonthlyCharges'

fig3 = px.histogram(
    df_plot,
    x=charges_col,
    color=churn_col,
    barmode='overlay',
    opacity=0.65,
    nbins=50,
    title='Monthly Charges Distribution by Churn Status',
    labels={charges_col: 'Monthly Charges ($)', churn_col: 'Churned?'},
    color_discrete_map={'Yes': '#EF553B', 'No': '#636EFA'}
)
fig3.update_layout(plot_bgcolor='white', height=400)
fig3.show()

## 5. Preprocessing Pipeline

Now we pass the raw data through our standardised preprocessing pipeline:
- `clean()` — handles type coercions, encodes the target variable, and removes duplicates
- `add_features()` — engineers derived features such as tenure bands, charge-per-month ratios, and interaction terms

We record shape before and after to confirm no rows were unexpectedly dropped.

In [ ]:
print(f'Shape BEFORE cleaning: {df_raw.shape}')

df_clean = clean(df_raw)
print(f'Shape AFTER clean():   {df_clean.shape}')

df_feat = add_features(df_clean)
print(f'Shape AFTER add_features(): {df_feat.shape}')

new_cols = [c for c in df_feat.columns if c not in df_raw.columns]
print(f'\nNew columns added ({len(new_cols)}): {new_cols}')

## 6. Correlation Heatmap of Numeric Features

Understanding correlations between numeric features helps us anticipate multicollinearity issues in linear models and gives intuition about which variables carry overlapping information. Strong correlations between `tenure` and `TotalCharges` are expected — longer-tenured customers have simply paid more in aggregate.

In [ ]:
numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()

# Limit to a manageable subset if there are many features
if len(numeric_cols) > 20:
    numeric_cols = numeric_cols[:20]

corr_matrix = df_feat[numeric_cols].corr()

fig_corr, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## 7. Save Processed Data

We persist the processed DataFrame to disk so that all downstream notebooks start from the same clean, feature-enriched baseline — avoiding the need to rerun the pipeline in every notebook.

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/customers_processed.csv'

df_feat.to_csv(output_path, index=False)
print(f'Processed data saved to: {output_path}')
print(f'Final shape: {df_feat.shape}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

## Key Findings

After this exploratory pass, four patterns stand out clearly:

1. **Contract type is the single strongest churn predictor.** Month-to-month customers churn at dramatically higher rates than one- or two-year contract holders. Any retention strategy should start with incentivising contract upgrades.

2. **Churn is heavily front-loaded.** Customers in their first 6–12 months show the highest churn rates. Once a customer passes the 24-month mark they are substantially 'sticky'. This points to the critical importance of the onboarding and early-engagement experience.

3. **Higher monthly charges correlate with elevated churn risk.** Churned customers skew toward the higher end of the `MonthlyCharges` distribution, suggesting that perceived value-for-money is a friction point for premium-tier customers.

4. **Tenure and TotalCharges are highly collinear** (expected), and both will need careful handling in linear models to avoid inflating coefficient estimates. Tree-based models handle this naturally.